In [1]:
import os
import json
import torch
import pickle
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

JSON_PATH = r'D:\Big_project_2025\CODE_THTN\products_final.json'
TEXT_MODEL_PATH = r'D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1'
TEXT_EMBEDDING_FILE = "moho_embeddings_text.pkl"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Device: cpu


In [2]:
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    products_data = json.load(f)

print("Total products:", len(products_data))

Total products: 193


In [3]:
if not os.path.exists(TEXT_EMBEDDING_FILE):
    print("🚀 Extracting text embeddings...")

    model = SentenceTransformer(TEXT_MODEL_PATH, device=device)

    data = []
    BATCH = 64

    for item in tqdm(products_data):
        p_id = item['product_id']

        texts = [
            item['content'].get('name', ''),
            item['content'].get('material', ''),
            item['ai_metadata'].get('full_description', '')
        ]

        labels = ["name", "material", "description"]

        vectors = model.encode(texts, show_progress_bar=False)

        for vec, label in zip(vectors, labels):
            if vec is not None:
                data.append({
                    "vector": vec,
                    "product_id": p_id,
                    "field": label
                })

    with open(TEXT_EMBEDDING_FILE, "wb") as f:
        pickle.dump(data, f)

    print("✅ Saved:", TEXT_EMBEDDING_FILE)

else:
    print("📂 Loading embeddings...")
    with open(TEXT_EMBEDDING_FILE, "rb") as f:
        data = pickle.load(f)

print("Total vectors:", len(data))

🚀 Extracting text embeddings...


The tokenizer you are loading from 'D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
100%|██████████| 193/193 [01:04<00:00,  2.98it/s]

✅ Saved: moho_embeddings_text.pkl
Total vectors: 579


In [ ]:
# 1. Chuyển đổi danh sách các vector thô thành một Ma trận (Tensor) duy nhất.
# Sử dụng kiểu dữ liệu float32 và đẩy lên thiết bị tính toán (CPU hoặc GPU).
# Việc chuyển sang Tensor giúp máy tính có thể tính toán hàng trăm vector cùng một lúc thay vì từng cái một.
text_vectors = torch.tensor([x["vector"] for x in data], dtype=torch.float32).to(device)

# 2. Chuẩn hóa (Normalize) các vector về độ dài bằng 1 (L2 normalization).
# Điều này cực kỳ quan trọng: Khi các vector đã được chuẩn hóa, phép tính "Độ tương đồng Cosine" 
# phức tạp sẽ trở thành phép nhân "Dot Product" đơn giản. 
# Phép toán này chạy cực nhanh trên các chip máy tính hiện đại.
text_vectors = F.normalize(text_vectors, p=2, dim=1)

# 3. Lưu lại thông tin mô tả (metadata) tương ứng.
# Biến này chứa thông tin bổ trợ như: vector này thuộc product_id nào và là của trường (field) nào.
text_meta = data

# 4. Kiểm tra kích thước ma trận cuối cùng.
print("Shape:", text_vectors.shape)  # Total vectors: 579 là vì lấy 3 trường cho 193 sản phẩm (193*3=579)

Shape: torch.Size([579, 512])


C:\Users\Admin\AppData\Local\Temp\ipykernel_29048\2980790436.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  text_vectors = torch.tensor([x["vector"] for x in data], dtype=torch.float32).to(device)


In [5]:
import pickle

# Đường dẫn đến file embedding văn bản
TEXT_EMBEDDING_FILE = "moho_embeddings_text.pkl"

# 1. Tải dữ liệu từ file pickle
with open(TEXT_EMBEDDING_FILE, "rb") as f:
    text_data = pickle.load(f)

# 2. Tìm các vector có product_id là "1"
target_id = "1"
results = [item for item in text_data if str(item['product_id']) == target_id]

# 3. Hiển thị kết quả
if results:
    print(f"--- Thông tin Text Embedding cho Product ID: {target_id} ---")
    print(f"Tìm thấy {len(results)} trường dữ liệu đã được số hóa.\n")
    
    for i, res in enumerate(results):
        field_name = res['field']
        vector = res['vector']
        
        print(f"Trường thứ {i+1}: [{field_name}]")
        print(f"- Kích thước vector (Shape): {vector.shape}")
        # Hiển thị 5 giá trị số đầu tiên để kiểm tra
        print(f"- 5 giá trị đầu tiên của vector: {vector[:5]}")
        print("-" * 30)
else:
    print(f"❌ Không tìm thấy thông tin văn bản cho ID '{target_id}' trong file.")

--- Thông tin Text Embedding cho Product ID: 1 ---
Tìm thấy 3 trường dữ liệu đã được số hóa.

Trường thứ 1: [name]
- Kích thước vector (Shape): (512,)
- 5 giá trị đầu tiên của vector: [-0.08132657  0.0855962   0.08720951 -0.05553775 -0.08283642]
------------------------------
Trường thứ 2: [material]
- Kích thước vector (Shape): (512,)
- 5 giá trị đầu tiên của vector: [ 0.01296821 -0.16392462  0.15821227  0.04029172  0.05691731]
------------------------------
Trường thứ 3: [description]
- Kích thước vector (Shape): (512,)
- 5 giá trị đầu tiên của vector: [-0.0201278   0.09802232  0.17927161  0.01259151  0.11580746]
------------------------------


# query

In [1]:
import torch
import pickle
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

# CONFIG
TEXT_EMBEDDING_FILE = "moho_embeddings_text.pkl"
TEXT_MODEL_PATH = r'D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1'

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# LOAD EMBEDDINGS
print("📂 Loading embeddings...")
with open(TEXT_EMBEDDING_FILE, "rb") as f:
    data = pickle.load(f)

# vector matrix
text_vectors = torch.tensor([x["vector"] for x in data], dtype=torch.float32).to(device)

# normalize trước
text_vectors = F.normalize(text_vectors, p=2, dim=1)

text_meta = data

print("✅ Loaded:", len(text_meta), "vectors")

# LOAD MODEL
print("🤖 Loading model...")
model = SentenceTransformer(TEXT_MODEL_PATH, device=device)

def search_text(query, top_k=5):
    query_emb = model.encode(query, convert_to_tensor=True).to(device)
    query_emb = F.normalize(query_emb, p=2, dim=0)

    scores = torch.matmul(text_vectors, query_emb)
    topk = torch.topk(scores, k=top_k * 5)

    best_results = {}

    for score, idx in zip(topk.values, topk.indices):
        item = text_meta[idx]
        pid = item["product_id"]
        field = item["field"]
        score_val = float(score)

        # nếu chưa có hoặc score tốt hơn → update
        if pid not in best_results or score_val > best_results[pid]["score"]:
            best_results[pid] = {
                "score": score_val,
                "field": field
            }

    # sort
    results = sorted(
        [(pid, v["score"], v["field"]) for pid, v in best_results.items()],
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    return results

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Device: cpu
📂 Loading embeddings...
✅ Loaded: 579 vectors
🤖 Loading model...


C:\Users\Admin\AppData\Local\Temp\ipykernel_12584\3455125258.py:19: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  text_vectors = torch.tensor([x["vector"] for x in data], dtype=torch.float32).to(device)
The tokenizer you are loading from 'D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [7]:
while True:
    query = input("\n🔍 Nhập query (Enter để thoát): ")

    if query.strip() == "":
        break

    results = search_text(query, top_k=5)

    print("\n🎯 Kết quả:")
    for pid, score, field in results:
        print(f"Product: {pid} | Score: {score:.4f} | Field: {field}")


🎯 Kết quả:
Product: 20 | Score: 0.9528 | Field: name
Product: 23 | Score: 0.9467 | Field: name
Product: 24 | Score: 0.9460 | Field: name
Product: 42 | Score: 0.9372 | Field: name
Product: 140 | Score: 0.9359 | Field: name
